## Exercise 1: Set Up the InsureGuard Data Platform

Before building the quality pipeline, you need to create the Unity Catalog structure and load the raw claims data. All cells in this exercise are provided — simply run them in order.

### ✅ Provided: Create the Unity Catalog structure

Run the cell below to create:
- A catalog named `insureguard_lab`
- A **bronze** schema for raw ingested data
- A **silver** schema for validated, analytics-ready data
- A managed volume `raw_files` in `insureguard_lab.bronze` as the landing zone for CSV files

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS insureguard_lab
  COMMENT 'InsureGuard — insurance claims quality platform';

CREATE SCHEMA IF NOT EXISTS insureguard_lab.bronze
  COMMENT 'Raw ingested claims data — as received from source systems';

CREATE SCHEMA IF NOT EXISTS insureguard_lab.silver
  COMMENT 'Validated claims data — ready for analytics and reporting';

CREATE VOLUME IF NOT EXISTS insureguard_lab.bronze.raw_files
  COMMENT 'Landing zone for raw claims CSV files';

### ✅ Provided: Load sample claims data into the volume

Run the cell below to write `insurance_claims_raw.csv` into your volume. This file contains **25 insurance claims** with a variety of intentional data quality issues:

| Issue type | Examples in the data |
|---|---|
| Null identifiers | Missing `claim_id` (row 6), missing `policy_id` (CLM-004, CLM-015, CLM-023) |
| Null amounts | Missing `claim_amount` (CLM-005, CLM-017) |
| Invalid dates | `"not-a-date"` (CLM-007), `"26/02/2026"` wrong format (CLM-020), `"invalid-date"` (CLM-014) |
| Non-numeric amounts | `"N/A"` as amount (CLM-008) |
| Negative amounts | `-500.00`, `-900.00`, `-1200.00` (CLM-009, CLM-014, CLM-024) |
| Invalid claim type | `"VEHICLE"` instead of allowed values (CLM-010) |
| Invalid status | `"INVALID_STATUS"`, `"INVALID"` (CLM-009, CLM-024) |

You will detect and handle each of these issues using pipeline expectations across Exercises 2–5.

In [ ]:
# ✅ Run this cell — it writes the raw claims CSV to your volume

claims_csv = """claim_id,policy_id,customer_id,claim_date,incident_date,claim_amount,claim_type,status,coverage_amount,agent_code
CLM-001,POL-1001,CUST-001,2026-01-05,2026-01-03,4500.00,AUTO,OPEN,25000.00,AGT-01
CLM-002,POL-1002,CUST-002,2026-01-07,2026-01-06,12000.00,HOME,PENDING,150000.00,AGT-02
CLM-003,POL-1003,CUST-003,2026-01-10,2026-01-09,250.00,HEALTH,CLOSED,5000.00,AGT-01
CLM-004,,CUST-004,2026-01-12,2026-01-11,8900.00,AUTO,OPEN,30000.00,AGT-03
CLM-005,POL-1005,CUST-005,2026-01-14,2026-01-13,,LIFE,PENDING,200000.00,AGT-02
,POL-1006,CUST-006,2026-01-15,2026-01-14,3200.00,HOME,CLOSED,80000.00,AGT-04
CLM-007,POL-1007,CUST-007,not-a-date,2026-01-16,7600.00,AUTO,OPEN,35000.00,AGT-03
CLM-008,POL-1008,CUST-008,2026-01-18,2026-01-17,N/A,HEALTH,PENDING,10000.00,AGT-01
CLM-009,POL-1009,CUST-009,2026-01-20,2026-01-19,-500.00,HOME,INVALID_STATUS,120000.00,AGT-04
CLM-010,POL-1010,CUST-010,2026-01-22,2026-01-21,15000.00,VEHICLE,CLOSED,50000.00,AGT-02
CLM-011,POL-1011,CUST-011,2026-01-25,2026-01-24,1800.00,AUTO,PENDING,20000.00,AGT-05
CLM-012,POL-1012,CUST-012,2026-01-27,2026-01-26,6400.00,HEALTH,OPEN,15000.00,AGT-01
CLM-013,POL-1013,CUST-013,2026-01-28,2026-01-27,92000.00,LIFE,CLOSED,300000.00,AGT-03
CLM-014,POL-1014,CUST-014,2026-01-30,invalid-date,-900.00,AUTO,PENDING,22000.00,AGT-02
CLM-015,,CUST-015,2026-02-01,2026-01-31,3300.00,HOME,OPEN,90000.00,AGT-05
CLM-016,POL-1016,CUST-016,2026-02-03,2026-02-02,5500.00,AUTO,CLOSED,28000.00,AGT-04
CLM-017,POL-1017,CUST-017,2026-02-05,2026-02-04,,HEALTH,UNDER_REVIEW,8000.00,AGT-01
CLM-018,POL-1018,CUST-018,2026-02-07,2026-02-06,44000.00,LIFE,PENDING,250000.00,AGT-03
CLM-019,POL-1019,CUST-019,2026-02-09,2026-02-08,2100.00,HOME,OPEN,70000.00,AGT-02
CLM-020,POL-1020,CUST-020,26/02/2026,2026-02-19,19000.00,AUTO,CLOSED,40000.00,AGT-05
CLM-021,POL-1021,CUST-021,2026-02-14,2026-02-13,780.00,HEALTH,PENDING,12000.00,AGT-04
CLM-022,POL-1022,CUST-022,2026-02-16,2026-02-15,33000.00,HOME,OPEN,200000.00,AGT-01
CLM-023,,CUST-023,2026-02-18,2026-02-17,8800.00,AUTO,CLOSED,32000.00,AGT-03
CLM-024,POL-1024,CUST-024,2026-02-20,2026-02-19,-1200.00,LIFE,INVALID,500000.00,AGT-02
CLM-025,POL-1025,CUST-025,2026-02-22,2026-02-21,5600.00,HOME,UNDER_REVIEW,110000.00,AGT-05"""

volume_path = "/Volumes/insureguard_lab/bronze/raw_files"
import os
os.makedirs(f"{volume_path}/claims", exist_ok=True)

with open(f"{volume_path}/claims/insurance_claims_raw.csv", "w") as f:
    f.write(claims_csv)

print(f"✅ Written: {volume_path}/claims/insurance_claims_raw.csv")
print(f"   Rows: {claims_csv.count(chr(10))} (including header)")

### ✅ Provided: Import the pipeline SDK

The cell below imports the `databricks.sdk.pipelines` package, which provides the `@dp.table`, `@dp.expect`, `@dp.expect_or_drop`, and `@dp.expect_or_fail` decorators used in Exercises 2–5.

Run this cell before defining any pipeline tables. Because this notebook is configured as **source code for a Lakeflow Spark Declarative Pipeline**, all decorated functions are picked up automatically when the pipeline runs.

> **Go to the setup lab file now.** Before continuing with Exercise 2, follow the instructions in the lab markdown to:
> 1. Create the Lakeflow Spark Declarative Pipeline in the Databricks UI
> 2. Connect this notebook as the pipeline source
> 3. Connect the notebook to the pipeline so you can run and validate from here

In [ ]:
# ✅ Run this cell — imports the pipeline decorator API
import databricks.sdk.pipelines as dp

---

## Exercise 2: Ingest Claims with Nullability Monitoring

Every claim that enters the InsureGuard platform must have a `claim_id` and a `policy_id`. Missing either of these makes the record untraceable and unusable for downstream analytics. A missing `claim_amount` is less critical for ingestion — it might arrive from a delayed assessment — but you still want to track it.

In this exercise, you define the **bronze ingestion table** that reads raw claims from the landing volume using Auto Loader. You add **nullability expectations** to monitor data quality:

| Column | Rule | Action |
|---|---|---|
| `claim_id` | Must not be null | Fail the pipeline update |
| `policy_id` | Must not be null | Drop the record |
| `claim_amount` | Must not be null | Warn (log, but let it through) |

> Use `@dp.expect_or_fail` for critical uniqueness requirements, `@dp.expect_or_drop` to silently remove unusable records, and `@dp.expect` to monitor issues you want to track without blocking data flow.

### 🧩 Task 2.1 — Define the bronze claims ingestion table

Write a `@dp.table` function named `bronze_claims_raw` that:

1. Reads from the claims CSV files in `/Volumes/insureguard_lab/bronze/raw_files/claims/` using **Auto Loader** (`cloudFiles` format, `csv` sub-format).
2. Sets `cloudFiles.schemaLocation` to `/Volumes/insureguard_lab/bronze/raw_files/schema` so Auto Loader can track schema evolution.
3. Sets `header` to `true` and `inferSchema` to `true`.
4. Returns a **streaming DataFrame** (`readStream`).
5. Adds three nullability expectations using the correct decorator for each action in the table above.

> 🤖 **Databricks Assistant prompt:** *"Write a Lakeflow Spark Declarative Pipelines Python table definition that reads CSV files using Auto Loader from a Unity Catalog volume path, with expect_or_fail for a null claim_id, expect_or_drop for a null policy_id, and expect (warn) for a null claim_amount."*

In [ ]:
# TODO: Define the bronze_claims_raw streaming table with nullability expectations
# Hint: Stack decorator calls in this order (innermost first):
#   1. @dp.table(comment="...")
#   2. @dp.expect_or_fail(...)
#   3. @dp.expect_or_drop(...)
#   4. @dp.expect(...)
#   5. def bronze_claims_raw(): ...

### ▶️ Validate your pipeline

After writing `bronze_claims_raw`, **validate the pipeline from within the notebook** using the pipeline toolbar that appears at the top of the notebook (once it is connected). Validation checks the table definition for errors without executing a full pipeline run.

> Once validation succeeds, run the pipeline and view the **Data quality** tab for `bronze_claims_raw` in the Pipeline UI (see the lab markdown for detailed steps). You should see violations logged for the `no_null_policy_id` expectation.

---

## Exercise 3: Enforce Type and Range Validation (Silver)

The bronze layer captured all records (except those dropped or failed by nullability rules). Now you build the **silver validated layer** that enforces stricter rules:

| Column | Rule | Rationale | Action |
|---|---|---|---|
| `claim_date` | Must be castable to `DATE` | Prevents broken date parsing in downstream reports | Drop |
| `claim_amount` | Must be castable to `DECIMAL(10,2)` | `"N/A"` strings would silently become NULL | Drop |
| `claim_amount` | Must be > 0 (when not null) | Negative amounts indicate a data entry error | Drop |
| `claim_type` | Must be one of `AUTO`, `HOME`, `LIFE`, `HEALTH` | Only recognised claim categories are processed | Drop |
| `status` | Must be one of `OPEN`, `CLOSED`, `PENDING`, `UNDER_REVIEW` | Unrecognised statuses break downstream state machines | Drop |

Use `try_cast` inside expectation expressions to safely check whether string columns contain valid typed values.

### 🧩 Task 3.1 — Define the silver validated claims table

Write a `@dp.table` function named `silver_claims_validated` that:

1. Reads from `bronze_claims_raw` as a **streaming** source (`spark.readStream.table("bronze_claims_raw")`).
2. Uses `@dp.expect_or_drop` for all five rules listed in the table above.
3. Uses `try_cast(claim_date AS DATE)` and `try_cast(claim_amount AS DECIMAL(10,2))` in the type-check expressions.
4. For the range check, allows `NULL` amounts to pass (the nullability warning is in the bronze layer): use `claim_amount IS NULL OR claim_amount > 0`.
5. Adds a `comment` to the `@dp.table` decorator describing the purpose of this table.

> 🤖 **Databricks Assistant prompt:** *"Write a Lakeflow Spark Declarative Pipelines Python streaming table that reads from bronze_claims_raw and uses expect_or_drop decorators to: (1) validate claim_date is castable to DATE using try_cast, (2) validate claim_amount is castable to DECIMAL(10,2) using try_cast, (3) ensure claim_amount is positive or null, (4) restrict claim_type to a specific set of values, (5) restrict status to a specific set of values."*

In [ ]:
# TODO: Define the silver_claims_validated streaming table
# Hint: Use @dp.expect_or_drop for each of the 5 rules
# Hint: try_cast expression example: "try_cast(claim_date AS DATE) IS NOT NULL"
# Hint: enum check example: "claim_type IN ('AUTO', 'HOME', 'LIFE', 'HEALTH')"

### 🧩 Task 3.2 — Use `expect_all_or_drop` to simplify the rules

Having five separate decorator calls works, but Lakeflow Spark Declarative Pipelines also supports grouping rules into a **dictionary** and applying them all at once with `@dp.expect_all_or_drop`.

Define a rules dictionary named `claims_quality_rules` that contains all five rules from Task 3.1, then rewrite `silver_claims_validated` to use `@dp.expect_all_or_drop(claims_quality_rules)` instead of five individual decorators.

> 🤖 **Databricks Assistant prompt:** *"Rewrite a Lakeflow Spark Declarative Pipelines Python table definition to use expect_all_or_drop with a dictionary of rules instead of multiple individual expect_or_drop decorators."*

In [ ]:
# TODO: Define a claims_quality_rules dictionary and rewrite silver_claims_validated
# using @dp.expect_all_or_drop(claims_quality_rules)

# claims_quality_rules = {
#     "rule_name": "sql_expression",
#     ...
# }

---

## Exercise 4: Quarantine Invalid Records

Dropping bad records keeps your silver table clean, but the dropped data isn't gone — it just never reached the destination. For a regulated industry like insurance, you must be able to **account for every rejected record**, investigate the cause, and potentially reprocess them after the source system is corrected.

In this exercise, you define a `silver_claims_quarantine` streaming table that captures records that fail any of the quality checks. Both `silver_claims_validated` and `silver_claims_quarantine` read from the same `bronze_claims_raw` source — one keeps the good records, the other captures the bad ones.

### 🧩 Task 4.1 — Define the quarantine table

Write a `@dp.table` function named `silver_claims_quarantine` that:

1. Also reads from `bronze_claims_raw` as a streaming source.
2. Filters to return **only records that fail at least one** of the following checks:
   - `claim_date` cannot be cast to DATE
   - `claim_amount` cannot be cast to DECIMAL(10,2)
   - `claim_amount` is not null and not > 0 (i.e., it is zero or negative)
   - `claim_type` is not in the allowed set
   - `status` is not in the allowed set
3. Uses **no expectations** — this table should accept all invalid records without dropping them.
4. Add a `comment` that makes the purpose of this table clear to other engineers.

> **Hint:** The quarantine filter is the logical opposite of the validation filter. Think in terms of `NOT` conditions or use `OR` to collect all failure cases.

> 🤖 **Databricks Assistant prompt:** *"Write a Lakeflow Spark Declarative Pipelines Python streaming table that reads from bronze_claims_raw and filters to keep only records that fail at least one data quality check — including bad date formats, bad numeric formats, negative amounts, invalid enum values. Use try_cast inside the filter expression."*

In [ ]:
# TODO: Define silver_claims_quarantine
# Hint: Use .filter(...) on the streaming DataFrame
# Hint: The quarantine condition is the complement of the silver_claims_validated conditions
# Hint: Use col() or SQL expressions — remember try_cast must be expressed as a SQL string
#       inside filter() using expr(), e.g.: from pyspark.sql.functions import expr, col

### ▶️ Validate and run the pipeline

After defining `silver_claims_quarantine`, validate and run the pipeline. Then check:
- How many records landed in `silver_claims_validated`?
- How many landed in `silver_claims_quarantine`?

Run the query below to compare counts:

In [ ]:
%sql
SELECT 'silver_claims_validated' AS table_name, COUNT(*) AS record_count
  FROM insureguard_lab.silver.silver_claims_validated
UNION ALL
SELECT 'silver_claims_quarantine', COUNT(*)
  FROM insureguard_lab.silver.silver_claims_quarantine;

---

## Exercise 5: Protect Against Schema Drift with Rescued Data

The InsureGuard fraud team has started adding a `fraud_score` column to claims coming from their new detection system. Your pipeline wasn't built with that column in mind. Without protection, Auto Loader may fail or silently lose that data when the schema doesn't match.

Lakeflow Spark Declarative Pipelines + Auto Loader handle this with the **rescued data column** pattern: columns that don't match the expected schema are preserved in a special `_rescued_data` JSON column instead of being dropped or causing failures.

In this exercise, you update `bronze_claims_raw` to add schema drift protection.

### 🧩 Task 5.1 — Add rescued data column support to the bronze table

Update your `bronze_claims_raw` definition to:

1. Add the Auto Loader option `"cloudFiles.schemaEvolutionMode"` set to `"rescue"`.
2. Add the option `"rescuedDataColumn"` set to `"_rescued_data"`.

This ensures that any column appearing in the source CSV that is **not part of the inferred schema** (such as `fraud_score`) is captured in `_rescued_data` as a JSON string rather than causing a pipeline failure.

> 🤖 **Databricks Assistant prompt:** *"Update a Lakeflow Spark Declarative Pipelines Auto Loader streaming table definition to add schema evolution mode set to rescue, and add a rescuedDataColumn option so unexpected columns are preserved in a JSON column called _rescued_data."*

In [ ]:
# TODO: Update bronze_claims_raw to include rescued data column options
# Add these two Auto Loader options to the readStream:
#   .option("cloudFiles.schemaEvolutionMode", "rescue")
#   .option("rescuedDataColumn", "_rescued_data")
#
# Keep all existing nullability expectations unchanged.

### 🧩 Task 5.2 — Inspect rescued data after a schema drift event

After updating and running the pipeline with the rescued data option enabled, you can query `_rescued_data` to understand what unexpected columns arrived:

Write a SQL query against `insureguard_lab.bronze.bronze_claims_raw` that:
1. Selects `claim_id`, `policy_id`, and `_rescued_data`
2. Filters to only rows where `_rescued_data` is not null
3. Orders by `claim_id`

> In a real scenario, the rows with non-null `_rescued_data` indicate records where the source sent extra fields. You would review these to decide whether to extend your schema to include the new column.

> 🤖 **Databricks Assistant prompt:** *"Write a SQL query to inspect rescued data from an Auto Loader pipeline table, showing only rows where the _rescued_data column is not null."*

In [ ]:
# TODO: Write a SQL or PySpark query to inspect _rescued_data
# %sql
# SELECT ...